# E1 · Справедливое распределение благ и обязанностей

**Эксперимент.** Сравниваем ответы LLM и алгоритмов на одних и тех же матрицах
аддитивных оценок. Каждая матрица трактуется дважды: как полезности благ
(`goods`, больше — лучше) и как затраты на обязанности (`chores`, меньше — лучше).
Проверяем, как меняются справедливость и эффективность при смене этой трактовки.
Каждый предмет неделим и назначается ровно одному участнику; пустые наборы допустимы.

**Что запускается.** Ноутбук генерирует входы по настройкам из раздела 3,
выполняет алгоритмы, рассчитывает метрики и строит радары и графики по n и m.
При `RUN_MODELS = True` в сравнение добавляются новые ответы модели.

**Как читать графики:** формат Figure 1
[Cookson, Ebadian, Shah — AAAI 2026](https://ebadian.org/preprint/llm_fairness_cr.pdf).
Чёрная линия — измеренные средние; зелёные секторы — справедливость,
оранжевые — эффективность; внешний круг соответствует 100%.
Проценты у EF/EF1/PROP/MMS — степени приближения, а не доли успешных ответов.
Расширенная версия добавляет EFX, SD-EF1 и MNW.

**Запуск:** выберите ядро `.venv/bin/python`, затем **Run All**.
Если окружения ещё нет, выполните из терминала:

```bash
uv venv --seed .venv
uv pip install --python .venv/bin/python -e ".[notebook]"
```

[README](README.md) · [Формулы метрик](README.md#метрики) ·
[Промпты из статей](README.md#промпты-из-других-статей) ·
[Готовые рисунки](README.md#графики)

## 1. Генерация входов и парный дизайн

Конфигурация `configs/scaling.json` задаёт десять размеров от 3×6 до 10×50.
Для каждого `base_id` генератор создаёт матрицу в каждой из выбранных групп.

| Группа | Как генерируется матрица |
|---|---|
| `paper` — Dirichlet–Multinomial | Общий для участников вектор `p ~ Dirichlet(1,…,1)`; для каждого i независимо `T_i ~ Uniform{(50−λ)m,…,(50+λ)m}`, затем строка `w_i ~ Multinomial(T_i,p)`. По умолчанию λ=20. |
| `uniform` | Все веса независимо равномерны на целых {1,…,10}. |
| `r=2`, `r=5`, `r=10` | Общая маска `B_ig ~ Bernoulli(0.5)`, веса `r`, если B=1, иначе 1. Маска одинакова во всех группах r. |
| `no_ef` | Общие целые веса с суммой, не кратной n, и индивидуальные поправки. [Почему EF невозможна](README.md#матрицы-без-ef) |
| `no_efx` | Три типа работ и два типа участников по конструкции He и Tao. Для обязанностей EFX невозможна. Подходят специальные размеры, например 4×13, 5×18 и 10×31. [Описание конструкции](README.md#обязанности-без-efx) |

Используется `seed=20261109`; базовые матрицы, группы и размеры имеют
воспроизводимые потоки случайности. Каждый метод получает **одинаковый вход**.
Повтор модели — отдельный запрос; сначала усредняем повторы внутри экземпляра.

Для оценивания `v_i(S)=100·sum(w_ig, g∈S)/sum(w_ig, g∈M)`.
Для обязанностей формула та же, но оценивает затраты `c_i`.
Нули допустимы, сумма каждой строки положительна.

**Отличие от статьи:** наш генератор `paper` следует опубликованному
Dirichlet–Multinomial-дизайну, но затем строки нормируются до 100.
Это сохраняет индивидуальные критерии справедливости, однако меняет SW
и выбор MinSW/MaxSW. Поэтому воспроизводится форма диаграмм на наших данных,
а не численные результаты авторов. Увеличение λ после нормировки уже не
воспроизводит исходный эффект разного масштаба полезностей участников.

## 2. Алгоритмы и задача для модели

| Название | Что делает | Где применяется |
|---|---|---|
| **Round-Robin (RR)** | Участники A1,…,An ходят по кругу. На своём ходу выбирают самое полезное оставшееся благо или наименее затратную оставшуюся работу. При равенстве — меньший ID предмета. | Блага и обязанности |
| **MaxSW** — Maximum Social Welfare | Каждое благо получает участник с максимальной нормированной полезностью. Максимизирует сумму полезностей без ограничений справедливости. | Блага |
| **MinSW** — Minimum Social Welfare / Minimum Social Cost | Каждую работу получает участник с минимальной нормированной стоимостью. Минимизирует суммарные затраты без ограничений справедливости. | Обязанности |
| **MNW (exact)** — Maximum Nash Welfare | Полный перебор максимизирует произведение полезностей участников. При недостижимом положительном произведении алгоритм пропускается. | Блага |
| **EF1-opt (exact)** | Полный перебор сначала минимизирует максимальный остаток зависти после удаления одного предмета, затем максимизирует SW или минимизирует C. | Блага и обязанности; контрольный ориентир |

Прежняя «эвристика обязанностей» — это **MinSW**, она сохранена под правильным
именем. Round-Robin для chores теперь запускается отдельно.
Алгоритмы используют фиксированный порядок участников, без перебора перестановок.
MinSW/MaxSW могут нарушать EF1. EF1-opt не оптимизирует SD-EF1 или EFX.

**Два вопроса к модели.** В условии `free` модель сама решает, что считать
справедливым дележом. Условия `ef1`, `prop` и `welfare` задают конкретную
цель. Все варианты получают одну матрицу, одно правило нормировки и один
формат JSON. После ответа каждому распределению считаем полный набор метрик.

Codex CLI запускает каждый ответ в новой сессии.
При новом модельном прогоне нужны установленный CLI и `codex login`.

In [ ]:
from dataclasses import replace
import os
import tempfile
from html import escape
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "e1-matplotlib"))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown
from chores_e1 import (
    ExperimentConfig, ModelConfig, ReferenceConfig, Setting,
    generate_instances, make_prompt, run_experiment,
    summary_table, radar_table, scaling_table, plot_inputs, plot_results,
    plot_scaling, EXTENDED_METRICS, PROMPT_VARIANTS, summarize,
)

def percent_table(frame):
    return frame.apply(lambda column: column.map(lambda value: f"{value:.1%}" if pd.notna(value) else "н/д"))

## 3. Настройки и выбор режима

- `CONFIG_MODE = "file"` загружает настройки из `CONFIG_FILE`. Профиль
  `configs/scaling.json` даёт 1 000 матриц, по 100 на размер.
- `CONFIG_MODE = "manual"` использует настройки внутри ветки `manual` в ячейке ниже.
  Здесь можно менять размеры, группы, число матриц на группу, промпты и модель.

`RUN_MODELS = False` запускает только алгоритмы. При `True` используются модели
выбранного профиля. Для `scaling.json` это 8 000 запросов на одну модель.
`JOBS = 4` разрешает до четырёх запросов одновременно. `None` берёт `jobs`
из выбранного профиля, по умолчанию 1.

Для ручного прогона на контрпримерах He и Tao используйте в ветке `manual`
`SETTINGS = (Setting(4, 13), Setting(5, 18), Setting(10, 31))`,
`GROUPS = ("no_efx",)` и `N_BASE = 1`. Эти же размеры уже заданы в файле
`configs/no_efx.json`.

In [ ]:
CONFIG_MODE = "file"  # "file" или "manual"
RUN_MODELS = False
JOBS = None  # например, 4 для параллельного запуска
CONFIG_FILE = "configs/scaling.json"

if CONFIG_MODE == "file":
    profile = ExperimentConfig.load(CONFIG_FILE)
elif CONFIG_MODE == "manual":
    SETTINGS = (Setting(5, 15), )
    GROUPS = ("paper", "uniform", "r=2", "r=10", "no_ef")
    N_BASE = 3  # матриц на группу для каждого размера
    PROMPT_CONDITIONS = ("free",)

    profile = ExperimentConfig(
        settings=SETTINGS,
        groups=GROUPS,
        n_base=N_BASE,
        prompt_variants=PROMPT_CONDITIONS,
        seed=20261109,
        paper_lambda=20,
        ratios=(2, 5, 10),
        repeats=1,
        row_total=100.0,
        reference=ReferenceConfig(max_allocations=10_000, chunk_size=2048),
        models=(ModelConfig(label="Codex", model="gpt-6.1-sol"),),
        output_dir="runs",
    )
else:
    raise ValueError('CONFIG_MODE должен быть "file" или "manual"')

config = replace(profile, models=profile.models if RUN_MODELS else (),
                 jobs=profile.jobs if JOBS is None else JOBS)
matrices_per_size = config.n_base * len(config.groups)
matrix_count = len(config.settings) * matrices_per_size
print(f"Матриц — {matrix_count} · На размер — {matrices_per_size} · На группу в размере — {config.n_base}")
print(f"Размеры — {[s.label for s in config.settings]}")
print("Режим — алгоритмы и модель" if config.models else "Режим — только алгоритмы")
for model in config.models:
    print(f"Метка {model.label} · Модель {model.model} · reasoning_effort={model.reasoning_effort}")
if config.models:
    n_calls = matrix_count * 2 * len(config.models) * config.repeats * len(config.prompt_variants)
    print(f"Планируется запросов к модели — {n_calls}")
    print(f"Одновременных запросов — до {min(config.jobs, n_calls)}")

## 4. Пример входа и полные промпты E1

На рисунке оценки нормированы до 100. В промпт входят исходные веса и правило
нормировки. Раскройте нужный вариант, чтобы увидеть точный текст для благ
и обязанностей.

По ответам на `free` мы видим свойства дележа. У одного дележа может быть
несколько свойств, поэтому мотив модели нельзя вывести однозначно.

In [ ]:
input_figure = plot_inputs(config)
display(input_figure)
plt.close(input_figure)

sample = generate_instances(config)[0]
for variant in config.prompt_variants:
    prompts = []
    for kind, title in (("goods", "Блага"), ("chores", "Обязанности")):
        prompt = make_prompt(sample.weights, kind, row_total=config.row_total, variant=variant)
        prompts.append(f"<p><strong>{title}</strong></p><pre>{escape(prompt)}</pre>")
    display(HTML(f"<details><summary>{PROMPT_VARIANTS[variant]} ({variant})</summary>"
                 + "".join(prompts) + "</details>"))

## 5. Выполнение и сохранение

Точный перебор ограничен `config.reference.max_allocations`.
При превышении лимита точные ориентиры, MMS, PO и MNW отсутствуют;
в графиках это «н/д», а не ноль.
Каждое выполнение создаёт новую папку в `runs/` с журналом и результатами.

In [ ]:
def progress(position, total, _key):
    if position % 100 == 0 or position == total:
        print(f"Завершено шагов: {position}/{total}", flush=True)

result = run_experiment(config, progress=progress if config.models else None)
print(f"Артефакты: {result.directory.resolve()}")
print(f"Распределений в сравнении: {len(result.rows)}")
print(f"Ответов модели: {(result.rows['source'] == 'model').sum()}")
skipped = (result.references["status"] == "skipped").sum()
if skipped:
    print(f"Точный ориентир пропущен для {skipped} задач; уменьшите размер или увеличьте лимит перебора.")

## 6. Метрики: приближения и точное выполнение

Все приближения ограничены [0,1]; **выше — лучше**.

| Ось | Смысл 100% |
|---|---|
| PROP | Каждый получает не меньше 1/n полной полезности благ или не больше 1/n стоимости обязанностей |
| EF | Никто не завидует чужому набору |
| EF1 | Зависть исчезает после удаления одного чужого блага / одной собственной работы |
| MMS | Выполнен maximin share для благ / minimax share для обязанностей, вычисленный полным перебором |
| PO | Нельзя одновременно улучшить полезности / затраты всех участников на общий мультипликативный коэффициент |
| SW | Достигнута максимальная суммарная полезность / минимальная стоимость, **без** ограничения EF1 |
| MNW | Достигнут максимальный геометрический средний welfare благ |
| EFX | Зависть исчезает после удаления **любого положительного** предмета с нужной стороны — вариант EFX+ |
| SD-EF1 | Доля ответов, удовлетворяющих EF1 на каждом положительном пороге оценок |

Для благ, например, `EF1-alpha = min(1, min_ij v_i(A_i) /
[v_i(A_j) − max_g∈A_j v_i(g)])`.
`SW = sum_i v_i(A_i) / sum_g max_i v_i(g)`.
Для обязанностей отношения обращаются, чтобы 100% всегда означало лучший результат:
`SW = min_B C(B)/C(A)`.
MNW для chores не определяем: максимизация произведения затрат не является
аналогом повышения благосостояния.

**PO-alpha и точная PO различаются:** первая оценивает общее мультипликативное
улучшение, вторая запрещает улучшение хотя бы одному при отсутствии ухудшений
остальным. Поле `po` хранит вторую проверку.
Формула PO-alpha зафиксирована в [разделе с метриками](README.md#метрики):
её буквальное совпадение с кодом авторов не подтверждено, поскольку доступная
расширенная версия статьи не содержит этой формулы.

Неверный JSON или неполное назначение учитывается как 0 в долях успеха
и приближениях; `d` и `total` для него не определяются.
«н/д» обозначает неприменимую или невычисленную метрику.
Бутстрэп-интервалы сохраняются в CSV; на радаре показаны средние без интервалов.

In [ ]:
radar_stats = radar_table(result)
wide = radar_stats.pivot(index=["setting", "method", "kind"], columns="metric", values="mean")
display(percent_table(wide.reindex(columns=list(EXTENDED_METRICS))))
radar_stats.to_csv(result.directory / "radar_summary.csv", index=False)

full_table = summary_table(result.rows)
full_table.to_csv(result.directory / "summary.csv")
rates = full_table.reset_index().groupby(["setting", "method", "kind"], sort=False)[
    ["valid", "ef1", "efx", "sd_ef1", "po"]
].mean()
display(Markdown("**Отдельно: доли точного выполнения свойств по всем группам**"))
display(percent_table(rates))
summarize(result.rows, n_base=config.n_base, seed=config.seed+1).to_csv(
    result.directory / "bootstrap.csv", index=False,
)

## 7. Шесть осей как на Figure 1 статьи

Одна панель — один метод. Блага и обязанности показаны отдельно, в одинаковом
масштабе 0–100%. Проценты вычислены по нашим распределениям. EF1-opt остаётся
в таблицах; чтобы добавить его на рисунки, передайте `include_reference=True`.
Здесь показаны радары для первого размера из настроек. Графики по n и m включают все размеры.

In [ ]:
radar_result = replace(result, config=replace(config, settings=config.settings[:1]))
paper_figures = plot_results(radar_result)
for index, figure in enumerate(paper_figures):
    for extension in ("png", "pdf", "svg"):
        figure.savefig(result.directory / f"paper_radar_{index+1}.{extension}", dpi=180, bbox_inches="tight")
    display(figure)
    plt.close(figure)

## 8. Расширенные диаграммы: EFX, SD-EF1 и MNW

Здесь сохранены шесть исходных критериев и добавлены три новых.
SD-EF1 — доля точных успехов; другие оси — непрерывные приближения.
Для chores MNW отмечена «н/д», поэтому чёрная линия на этом участке прерывается.
В графиках ноутбука все группы имеют равный вес. Отдельную группу можно
выбрать в фильтре «Тип входов» на [интерактивном сайте](https://evgeniizhidkov.github.io/delezhii_project/).

In [ ]:
extended_figures = plot_results(radar_result, metrics=EXTENDED_METRICS)
for index, figure in enumerate(extended_figures):
    for extension in ("png", "pdf", "svg"):
        figure.savefig(result.directory / f"extended_radar_{index+1}.{extension}", dpi=180, bbox_inches="tight")
    display(figure)
    plt.close(figure)

## 9. Как меняются результаты с размером матрицы

На рисунке в стиле рис. 3 слева меняется `n` при `m = 15`, справа — `m`
при `n = 5`. Вертикальные отрезки показывают 95%-е бутстрэп-интервалы.
Графики для всех размеров тоже сохраняются в папке прогона. Строки показывают
EF1, эффективность, SD-EF1 и EFX+. Для благ добавлена нижняя оценка доли MNW
для больших матриц. `SCALING_GROUP = "no_ef"` оставляет только входы без EF.

In [ ]:
SCALING_GROUP = None
scaling_stats = scaling_table(result, group=SCALING_GROUP)
scaling_stats.to_csv(result.directory / "scaling_summary.csv", index=False)
figures = plot_scaling(result, group=SCALING_GROUP, stats=scaling_stats)
paper_figures = plot_scaling(result, group=SCALING_GROUP, paper=True, stats=scaling_stats)
for name, figure in (figures | paper_figures).items():
    figure.savefig(result.directory / f"{name}.png", dpi=160)
    if name in paper_figures or not paper_figures:
        display(figure)
    plt.close(figure)

## 10. Контрпримеры для EFX при распределении обязанностей

Здесь показаны результаты для группы `no_efx`. Если её нет в основном прогоне,
ноутбук отдельно проверяет три матрицы He и Tao на 4, 5 и 10 участников.
По теореме 1 точной EFX на этих входах достичь невозможно. В таблице показаны
оценки алгоритмов по EF1, SD-EF1 и EFX+. [Конструкция He и Tao](https://arxiv.org/html/2606.08872v2#A1)

In [ ]:
if "no_efx" in config.groups:
    stress_result = result
else:
    stress_result = run_experiment(ExperimentConfig.load("configs/no_efx.json"))
columns = ["setting", "method", "ef1", "sd_ef1", "efx", "efx_alpha"]
chore_cases = (stress_result.rows["kind"] == "chores") & (stress_result.rows["group"] == "no_efx")
display(stress_result.rows.loc[chore_cases, columns])

## 11. Промпты в других статьях

Ниже — примеры дизайнов, с которыми можно сопоставить варианты E1.

### Источники и протоколы

Проверено 14 ноября 2026 года. Ниже краткие цитаты, описание протокола
и наши адаптированные схемы. Схемы не являются дословными промптами авторов.

### Cookson, Ebadian, Shah — Fairness Perceptions of Large Language Models

AAAI 2026, 40(42):35393–35401.
[Опубликованный PDF, §2 и §6](https://ebadian.org/preprint/llm_fairness_cr.pdf) ·
[DOI](https://doi.org/10.1609/aaai.v40i42.40848) ·
[Расширенный PDF, Appendix A](https://www.cs.toronto.edu/~nisarg/papers/llm-fairness.pdf).

Протокол: контекст → цель → оценки → JSON. В опубликованной версии цели —
общая справедливость, математически заданная EF1 и эквивалентная комбинаторная
задача без языка справедливости. Меняются также контекст и ориентация таблицы.
Ответ сопоставляет предмету ID получателя, без пояснений.

Короткая цитата общей цели из Appendix A:

> Using the people's scores, you should allocate the objects to the people in the fairest way possible.

Наша адаптированная схема для сравнения вариантов:

```text
Дано n участников, m неделимых предметов и матрица аддитивных полезностей.
[Цель: общий справедливый делёж / условие EF1 / только комбинаторное условие.]
[Матрица по участникам / транспонированная матрица по предметам.]
Назначь каждый предмет одному участнику. Верни JSON «предмет → получатель».
```

В доступном расширенном PDF Appendix A перечисляет также MNW/MUW-цели,
но это ранняя версия с другим генератором. Её нельзя смешивать с протоколом
опубликованной статьи. Наш E1 задаёт EF1 явно и добавляет вторичную цель SW/C.

### Hosseini, Khanna — Distributive Fairness in Large Language Models

Evaluating Alignment with Human Values, NeurIPS 2025.
[Статья, §3 и Appendix J.1](https://arxiv.org/html/2502.00313v2#A10.SS1) ·
[Сравнение шаблонов, Appendix G.2](https://arxiv.org/html/2502.00313v2#A7.SS2).

Два сообщения: сначала модель свободно выбирает и объясняет делёж,
затем отдельный запрос извлекает уже выбранное распределение в JSON.
Авторы допускают выбрасывание благ; есть задачи с деньгами. Это отличается
от полного назначения всех предметов в E1.

Цитата инструкции второго шага:

> Please present the allocation you have selected in the following JSON format:

Наша адаптированная схема:

```text
Шаг 1. Вот участники, предметы и их оценки. Какой делёж ты выберешь
как наиболее справедливый? Объясни выбор.
Шаг 2. Используя предыдущий ответ, запиши выбранный делёж как JSON.
Для неназначенного предмета используй null.
```

Полные примеры исходных сообщений приведены в J.1. В E1 используется
один запрос со строгой JSON-схемой; это различие следует учитывать
при сопоставлении результатов.

### Han et al. — Fair Like Us? Auditing LLM Alignment in Resource Allocation

2026, arXiv:2609.29692.
[Статья, §4 и Appendix C.2](https://arxiv.org/html/2609.29692#A3.SS2).

Модель выступает получателем уже заданного набора. T1–T5 меняют доступность
чужих наборов и оценок, а также вопрос: приемлема ли доля или хочется
обменять её на чужую. В основных сериях сохраняется история предыдущих
вопросов. Это оценка заданного распределения, а не его построение.

Цитата вопроса T1:

> Do you find your share acceptable? Please limit your response to either ’Yes.’ or ’No.’.

Наша адаптированная схема:

```text
Ты один из четырёх участников. Вот твои оценки предметов и твой набор.
[Дополнительно: чужие наборы и/или чужие оценки.]
[Вопрос: приемлема ли доля / какой обмен ты выбираешь?]
Ответь в указанном формате.
```

Эти примеры помогают сопоставить постановки разных исследований.
Полный промпт E1 для благ и обязанностей выводится в разделе 4.

## 12. Как интерпретировать результаты

В опубликованном прогоне пять входов 3×6, по одному на группу. Для каждого
входа модель решала две задачи с четырьмя промптами, всего 40 ответов.
Сравнивайте методы на одних и тех же таблицах. На пяти входах в каждой
постановке один ответ меняет долю выполнения на 20 процентных пунктов.

В [папке опубликованного прогона](runs/published/3x6_5matrices_gpt-6.1-sol_4prompts/) лежат графики, таблицы и журнал
опубликованного прогона.
Воспроизвести их без обращений к модели:

```bash
.venv/bin/python archive/scripts/build_revised_reports.py
```

[Все формулы и соглашения о нулях](README.md#метрики) ·
[Проверки и команды запуска](README.md#воспроизводимость)